# Promotion Impact Pipeline

**Today I'm going to show you** how we built a pipeline to answer one question:

> **When we run campaigns (Black Friday, Christmas, etc.), is the revenue lift real—or inflated by how we join the data?**

---

## The problem we're solving

We want to know: **Did promotions actually drive more revenue, or do the numbers just *look* better because of duplicates and messy joins?**

---

## Why this gets messy (three big risks)

1. **Same campaign, many channels** — Black Friday might show up in email, social, and display. Join orders to campaigns naively and one order becomes 3 rows → revenue gets counted 3 times.

2. **Many-to-many joins** — Orders have multiple items; campaigns have multiple channels. Join everything at once and rows explode. Sum revenue on that and you're way over.

3. **Data changes over time** — Raw data gets fixed or reloaded. Without a clear pipeline (raw → staging → curated), we can't tell if a number changed because of the business or because of how we built the tables.

---

## Project setup

- **Environment**: Python 3.10 in a Jupyter Notebook using the `duckdb` Python package (in-memory database for this notebook).  
- **Storage**: CSV source files under `data/raw`; when running the full pipeline outside the notebook we also use an on-disk DuckDB file via `build.py`.  
- **Libraries**: `duckdb`, `pandas`, and `pathlib.Path` for file paths.

## What we used

- **Olist Brazilian E-commerce data** (orders, order items, payments, etc.)  
- **Campaign metadata** we added: campaign windows, channels, discount rules

---

## What we're building

- A **join-safe** fact table: one clear grain, no row multiplication  
- So we can honestly answer: **Did campaigns lift revenue?**

---

## How we get there: Raw → Staging → Curated

| Layer | In one sentence |
|-------|------------------|
| **Raw** | Keep the evidence; don't change a thing. |
| **Staging** | Clean, dedupe, and pre-aggregate so joins don't explode. |
| **Curated** | Publish one clean fact table we can trust. |

We use **snapshot tables** (not views) so each run gives the same result until we deliberately rerun.

---

## Roadmap for the rest of the notebook

1. **Setup & raw load** — Idempotent load; raw_orders, raw_order_items, raw_campaigns  
2. **Staging** — Clean text, dedupe campaigns, pre-aggregate where needed  
3. **Curated** — Build the fact table with safe joins  
4. **Audits** — Checks that the pipeline didn't lose or multiply rows  
5. **Report** — Campaign vs no-campaign revenue (the answer we came for)

---
# Let's build it

**First:** connection, paths, and a couple of helpers so we can run SQL and see results.

In [1]:
import duckdb
import pandas as pd
from pathlib import Path

BASE_DIR = Path(".").resolve()
DATA_DIR = BASE_DIR / "data" / "raw"

con = duckdb.connect(":memory:")
con.execute("PRAGMA disable_progress_bar")

def q(sql: str) -> pd.DataFrame:
    return con.execute(sql).fetchdf()

## Step 1 — The evidence locker (raw data)

We load orders, order items, and campaigns **exactly as they are**. No filtering, no cleaning. This is our immutable starting point so we can always trace back.

In [2]:
# Raw layer: Python + DuckDB only (no external SQL files)
# Make this cell self-contained so it still works after a kernel restart.
if "DATA_DIR" not in globals():
    import duckdb
    import pandas as pd
    from pathlib import Path
    BASE_DIR = Path(".").resolve()
    DATA_DIR = BASE_DIR / "data" / "raw"
    con = duckdb.connect(":memory:")
    con.execute("PRAGMA disable_progress_bar")
    def q(sql: str) -> pd.DataFrame:
        return con.execute(sql).fetchdf()

for name, fname in [
    ("raw_orders", "olist_orders_dataset.csv"),
    ("raw_order_items", "olist_order_items_dataset.csv"),
    ("raw_campaigns", "campaign_metadata.csv"),
]:
    path = DATA_DIR / fname
    if path.exists():
        con.execute(f"""
            CREATE OR REPLACE TABLE {name} AS
            SELECT * FROM read_csv_auto('{path.as_posix()}');
        """)
        print(f"Loaded {fname} -> {name}")
    else:
        print(f"Skipped {fname} (not found)")

Loaded olist_orders_dataset.csv -> raw_orders
Loaded olist_order_items_dataset.csv -> raw_order_items
Loaded campaign_metadata.csv -> raw_campaigns


In [3]:
print("Raw orders sample:")
display(q("SELECT * FROM raw_orders LIMIT 5"))

Raw orders sample:


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26


In [4]:
print("Raw campaigns (coupon windows, channels):")
display(q("SELECT * FROM raw_campaigns"))

Raw campaigns (coupon windows, channels):


,campaign_id,campaign_name,channel,window_start,window_end,discount_pct,coupon_code
0,C001,Black Friday 2017,email,2017-11-24,2017-11-26,15.0,BF2017
1,C001,Black Friday 2017,social,2017-11-24,2017-11-26,15.0,BF2017
2,C002,Cyber Monday 2017,email,2017-11-27,2017-11-28,10.0,CM2017
3,C002,Cyber Monday 2017,display,2017-11-27,2017-11-28,10.0,CM2017
4,C003,Christmas 2017,email,2017-12-15,2017-12-25,20.0,XMAS17
5,C003,Christmas 2017,social,2017-12-15,2017-12-25,20.0,XMAS17
6,C003,Christmas 2017,display,2017-12-15,2017-12-25,20.0,XMAS17
7,C004,New Year 2018,email,2017-12-28,2018-01-05,5.0,NY2018
8,C005,Carnival 2018,social,2018-02-09,2018-02-14,12.0,CARN18
9,C006,Black Friday 2018,email,2018-11-23,2018-11-25,18.0,BF2018


## Step 2 — Schemas for staging and curated

We create the staging and curated schemas so the next steps have a place to write their tables.

In [5]:
# Schema: Python + DuckDB only
con.execute("""CREATE SCHEMA IF NOT EXISTS staging; CREATE SCHEMA IF NOT EXISTS curated;""")
print("Created staging and curated schemas")

Created staging and curated schemas


## Step 3 — Staging: clean and dedupe

**What we do here:**

- **Orders** — Trim IDs, cast timestamps, and keep only real purchases (`delivered`, `shipped`, `in transit`, `processing`, `approved`). Cancelled and unavailable orders are dropped.
- **Order items** — Cast price and freight, drop rows with missing keys, and add `revenue_per_item = price + freight_value`.
- **Campaigns** — The metadata file has **one row per campaign per channel** (Black Friday appears for email *and* social). We keep **one row per campaign** with `ROW_NUMBER() OVER (PARTITION BY campaign_id …) = 1`. Without this, every order inside a campaign window would match 2–3 rows and its revenue would be counted 2–3 times.

**Takeaway:** By the time we join in the curated layer, each order can match at most one campaign row. No explosion.


In [6]:
# Staging: SQL inlined in notebook (Python + DuckDB only, no .sql files)
con.execute(r"""
CREATE OR REPLACE TABLE staging.orders AS
SELECT TRIM(order_id) AS order_id, TRIM(customer_id) AS customer_id, LOWER(TRIM(order_status)) AS order_status,
  CAST(order_purchase_timestamp AS TIMESTAMP) AS order_purchase_timestamp,
  CAST(order_approved_at AS TIMESTAMP) AS order_approved_at,
  CAST(order_delivered_customer_date AS TIMESTAMP) AS order_delivered_customer_date,
  CAST(order_estimated_delivery_date AS TIMESTAMP) AS order_estimated_delivery_date
FROM raw_orders
WHERE order_id IS NOT NULL AND order_purchase_timestamp IS NOT NULL
  AND LOWER(TRIM(order_status)) IN ('delivered', 'shipped', 'in transit', 'processing', 'approved')
""")
con.execute(r"""
CREATE OR REPLACE TABLE staging.order_items AS
SELECT TRIM(order_id) AS order_id, order_item_id, TRIM(product_id) AS product_id, TRIM(seller_id) AS seller_id,
  CAST(price AS DOUBLE) AS price, CAST(freight_value AS DOUBLE) AS freight_value,
  CAST(price AS DOUBLE) + CAST(freight_value AS DOUBLE) AS revenue_per_item
FROM raw_order_items WHERE order_id IS NOT NULL AND order_item_id IS NOT NULL
""")
con.execute(r"""
CREATE OR REPLACE TABLE staging.campaigns AS
SELECT campaign_id, campaign_name, window_start, window_end, discount_pct, coupon_code FROM (
  SELECT TRIM(campaign_id) AS campaign_id, LOWER(TRIM(campaign_name)) AS campaign_name,
    CAST(window_start AS DATE) AS window_start, CAST(window_end AS DATE) AS window_end,
    discount_pct, LOWER(TRIM(COALESCE(coupon_code, ''))) AS coupon_code,
    ROW_NUMBER() OVER (PARTITION BY TRIM(campaign_id) ORDER BY channel) AS rn
  FROM raw_campaigns WHERE campaign_id IS NOT NULL
) t WHERE rn = 1
""")
print("Staging tables created")

Staging tables created


In [7]:
print("staging.orders sample:")
display(q("SELECT * FROM staging.orders LIMIT 5"))

staging.orders sample:


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-10 21:25:13,2017-10-18
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-08-07 15:27:45,2018-08-13
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-17 18:06:29,2018-09-04
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-12-02 00:28:42,2017-12-15
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-16 18:17:02,2018-02-26


In [8]:
print("staging.order_items sample:")
display(q("SELECT * FROM staging.order_items LIMIT 5"))

staging.order_items sample:


,order_id,order_item_id,product_id,seller_id,price,freight_value,revenue_per_item
0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,58.90,13.29,72.19
1,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,239.90,19.93,259.83
2,000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,199.00,17.87,216.87
3,00024acbcdf0a6daa1e931b038114c75,1,7634da152a4610f1595efa32f14722fc,9d7a1d34a5052409006425275ba1c2b4,12.99,12.79,25.78
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,ac6c3623068f30de03045865e4e10089,df560393f3a51e74553ab94004ba5c87,199.90,18.14,218.04


Staging campaigns: one row per campaign (same campaign across email, social, display → one canonical row).

In [9]:
print("staging.campaigns (deduplicated by campaign_id):")
display(q("SELECT * FROM staging.campaigns"))

staging.campaigns (deduplicated by campaign_id):


,campaign_id,campaign_name,window_start,window_end,discount_pct,coupon_code
0,C002,cyber monday 2017,2017-11-27,2017-11-28,10.0,cm2017
1,C001,black friday 2017,2017-11-24,2017-11-26,15.0,bf2017
2,C005,carnival 2018,2018-02-09,2018-02-14,12.0,carn18
3,C006,black friday 2018,2018-11-23,2018-11-25,18.0,bf2018
4,C003,christmas 2017,2017-12-15,2017-12-25,20.0,xmas17
5,C004,new year 2018,2017-12-28,2018-01-05,5.0,ny2018


## Step 4 — Curated: the fact table (join-safe)

**Grain:** one row per **order** (`order_id`). Revenue is summed from the order items *before* the join, so an order with three items still produces exactly one row.

**How we join:**

- **Order items → order revenue** — pre-aggregated in a CTE (`GROUP BY order_id`), so items can never multiply order rows
- **Orders → order revenue** (INNER) — only orders that actually have items
- **Orders → campaigns** (LEFT) — the purchase date falls inside a campaign window; campaign windows don't overlap and campaigns are deduplicated in staging, so each order gets at most one campaign
- **Orders → customers** (LEFT) — a surrogate key (`customer_sk`) from `customers_dim`

**Why we can trust it:** every join is either 1:1 or many-to-one into the order, so `SUM(order_revenue)` on this table counts each order once.


In [10]:
# Curated: SQL inlined in notebook (Python + DuckDB only)
con.execute(r"""
CREATE OR REPLACE TABLE curated.campaigns_dim AS
SELECT DENSE_RANK() OVER (ORDER BY campaign_id) AS campaign_sk, campaign_id, campaign_name,
  window_start, window_end, discount_pct, coupon_code FROM staging.campaigns
""")
con.execute(r"""
CREATE OR REPLACE TABLE curated.customers_dim AS
SELECT DENSE_RANK() OVER (ORDER BY customer_id) AS customer_sk, customer_id
FROM (SELECT DISTINCT customer_id FROM staging.orders WHERE customer_id IS NOT NULL) o
""")
con.execute(r"""
CREATE OR REPLACE TABLE curated.curated_revenue_fact AS
WITH order_revenue AS (
  SELECT order_id, SUM(price + freight_value) AS order_revenue,
    SUM(price) AS order_price_sum, SUM(freight_value) AS order_freight_sum
  FROM staging.order_items WHERE order_id IS NOT NULL GROUP BY order_id
)
SELECT o.order_id, o.order_purchase_timestamp, cd.customer_sk, camp.campaign_sk, camp.campaign_id,
  camp.campaign_name, camp.discount_pct AS campaign_discount_pct,
  ord.order_revenue, ord.order_price_sum, ord.order_freight_sum
FROM staging.orders o
INNER JOIN order_revenue ord ON o.order_id = ord.order_id
LEFT JOIN curated.campaigns_dim camp ON CAST(o.order_purchase_timestamp AS DATE) BETWEEN camp.window_start AND camp.window_end
LEFT JOIN curated.customers_dim cd ON o.customer_id = cd.customer_id
""")
print("curated.campaigns_dim, customers_dim, curated_revenue_fact created")

curated.campaigns_dim, customers_dim, curated_revenue_fact created


In [11]:
print("curated_revenue_fact sample (1 row = 1 order event):")
display(q("SELECT * FROM curated.curated_revenue_fact LIMIT 10"))

curated_revenue_fact sample (1 row = 1 order event):


,order_id,order_purchase_timestamp,customer_sk,campaign_sk,campaign_id,campaign_name,campaign_discount_pct,order_revenue,order_price_sum,order_freight_sum
0,00061f2a7bc09da83e415a52dc8a4af1,2018-03-24 22:16:10,76404,<NA>,None,None,NaN,68.87,59.99,8.88
1,0011d82c4b53e22e84023405fb467e57,2018-01-16 21:43:23,12276,<NA>,None,None,NaN,315.33,289.00,26.33
2,002175704e8b209f61b9ad5cfd92b60e,2018-04-22 12:13:25,63273,<NA>,None,None,NaN,123.11,109.90,13.21
3,00259a44fcad3fc0474329e925d14fc3,2017-12-27 17:52:11,45448,<NA>,None,None,NaN,34.09,19.99,14.10
4,002af7fa9eb99d1456bceb6a13d6fc0b,2018-02-18 18:46:28,71726,<NA>,None,None,NaN,109.45,88.34,21.11
5,00324b3eda39ba5ecce3945823e3594c,2018-04-10 01:14:18,78278,<NA>,None,None,NaN,110.07,76.00,34.07
6,004157daafa0bc8672e01f00e4f0c04f,2017-05-21 13:13:54,25226,<NA>,None,None,NaN,222.91,189.00,33.91
7,00418a49a685c6bb60633291c3fab17a,2017-10-03 20:43:03,47991,<NA>,None,None,NaN,56.78,49.00,7.78
8,0084e195fbd72ae51599af47f04afede,2018-03-14 23:57:00,72813,<NA>,None,None,NaN,113.03,89.90,23.13
9,00b2d2f2b5f7b98e6b1828764660134e,2018-08-09 08:27:54,67400,<NA>,None,None,NaN,159.69,146.01,13.68


## Step 5 — Audits: did we do it right?

Before we trust the numbers, we run **checks** (stacked with `UNION ALL`) and store them in `audit_results`:

- **No orphans** — every `campaign_sk` and `customer_sk` in the fact table exists in its dimension
- **Unique dimensions** — no duplicate `campaign_id` or `customer_id`
- **Grain check** — no duplicate `order_id` in the fact table (must be **0**)
- **Reconciliation** — the set of orders in curated equals the set of staged orders that have items, in both directions (nothing lost, nothing extra)

If any check fails, the pipeline stops. This is our quality gate.


In [12]:
# Checks: SQL inlined in notebook (Python + DuckDB only)
con.execute(r"""
CREATE OR REPLACE TABLE audit_results AS
SELECT 'no_orphans_campaign_sk' AS check_name, CASE WHEN COUNT(*) > 0 THEN 1 ELSE 0 END AS fail_count, CAST(COUNT(*) AS VARCHAR) AS detail
FROM curated.curated_revenue_fact f LEFT JOIN curated.campaigns_dim d ON f.campaign_sk = d.campaign_sk
WHERE f.campaign_id IS NOT NULL AND d.campaign_sk IS NULL
UNION ALL SELECT 'no_orphans_customer_sk', CASE WHEN COUNT(*) > 0 THEN 1 ELSE 0 END, CAST(COUNT(*) AS VARCHAR)
FROM curated.curated_revenue_fact f LEFT JOIN curated.customers_dim d ON f.customer_sk = d.customer_sk
WHERE d.customer_sk IS NULL AND f.customer_sk IS NOT NULL
UNION ALL SELECT 'campaigns_dim_unique', CASE WHEN cnt > 0 THEN 1 ELSE 0 END, CAST(COALESCE(cnt, 0) AS VARCHAR)
FROM (SELECT COUNT(*) AS cnt FROM (SELECT campaign_id FROM curated.campaigns_dim GROUP BY campaign_id HAVING COUNT(*) > 1) x)
UNION ALL SELECT 'customers_dim_unique', CASE WHEN cnt > 0 THEN 1 ELSE 0 END, CAST(COALESCE(cnt, 0) AS VARCHAR)
FROM (SELECT COUNT(*) AS cnt FROM (SELECT customer_id FROM curated.customers_dim GROUP BY customer_id HAVING COUNT(*) > 1) x)
UNION ALL SELECT 'fact_order_id_unique', CASE WHEN cnt > 0 THEN 1 ELSE 0 END, CAST(COALESCE(cnt, 0) AS VARCHAR)
FROM (SELECT COUNT(*) AS cnt FROM (SELECT order_id FROM curated.curated_revenue_fact GROUP BY order_id HAVING COUNT(*) > 1) x)
UNION ALL SELECT 'reconciliation_expected_not_in_curated', CASE WHEN cnt > 0 THEN 1 ELSE 0 END, CAST(COALESCE(cnt, 0) AS VARCHAR)
FROM (SELECT COUNT(*) AS cnt FROM (SELECT order_id FROM staging.orders WHERE order_id IN (SELECT order_id FROM staging.order_items) EXCEPT SELECT order_id FROM curated.curated_revenue_fact) x)
UNION ALL SELECT 'reconciliation_curated_not_in_expected', CASE WHEN cnt > 0 THEN 1 ELSE 0 END, CAST(COALESCE(cnt, 0) AS VARCHAR)
FROM (SELECT COUNT(*) AS cnt FROM (SELECT order_id FROM curated.curated_revenue_fact EXCEPT SELECT order_id FROM staging.orders WHERE order_id IN (SELECT order_id FROM staging.order_items)) x)
UNION ALL SELECT 'curated_revenue_fact_row_count', 0, CAST(COUNT(*) AS VARCHAR) FROM curated.curated_revenue_fact
UNION ALL SELECT 'campaign_touch_count', 0, CAST(COUNT(*) AS VARCHAR) FROM curated.curated_revenue_fact WHERE campaign_id IS NOT NULL
""")
audit_df = q("SELECT * FROM audit_results")
display(audit_df)

,check_name,fail_count,detail
0,no_orphans_campaign_sk,0,0
1,no_orphans_customer_sk,0,0
2,campaigns_dim_unique,0,0
3,customers_dim_unique,0,0
4,fact_order_id_unique,0,0
5,reconciliation_expected_not_in_curated,0,0
6,reconciliation_curated_not_in_expected,0,0
7,curated_revenue_fact_row_count,0,97887
8,campaign_touch_count,0,7016


In [13]:
# Fail-fast: same as build.py — exit if any check has fail_count > 0
fail_count = int(q("SELECT COALESCE(SUM(fail_count), 0) AS n FROM audit_results").iloc[0, 0])
if fail_count > 0:
    print("FAILED CHECKS:", audit_df[audit_df["fail_count"] > 0][["check_name", "fail_count"]].to_string())
    raise SystemExit(1)
print("ALL CHECKS PASSED")

ALL CHECKS PASSED


## Step 6 — The answer: campaign vs no-campaign revenue

Using the **curated** fact table (so no double-counting), we compare:

- Revenue during campaign windows  
- Revenue outside campaign windows  

This is the report we built the pipeline for.

In [14]:
# Same logic as build.py output: use curated_revenue_fact (1 row per order)
report = q("""
SELECT
    CASE WHEN campaign_id IS NOT NULL THEN 'campaign' ELSE 'no_campaign' END AS segment,
    COUNT(*) AS order_count,
    ROUND(SUM(order_revenue), 2) AS total_revenue,
    ROUND(AVG(order_revenue), 2) AS avg_revenue_per_order
FROM curated.curated_revenue_fact
GROUP BY 1
""")
display(report)

,segment,order_count,total_revenue,avg_revenue_per_order
0,campaign,7016,1045579.28,149.03
1,no_campaign,90871,14620959.00,160.90


In [15]:
report_by_campaign = q("""
SELECT
    campaign_name,
    campaign_discount_pct,
    COUNT(DISTINCT order_id) AS order_count,
    ROUND(SUM(order_revenue), 2) AS total_revenue
FROM curated.curated_revenue_fact
WHERE campaign_id IS NOT NULL
GROUP BY campaign_name, campaign_discount_pct
ORDER BY total_revenue DESC
""")
display(report_by_campaign)

,campaign_name,campaign_discount_pct,order_count,total_revenue
0,black friday 2017,15.0,2045,302412.31
1,christmas 2017,20.0,1513,239001.98
2,new year 2018,5.0,1409,207941.47
3,carnival 2018,12.0,1275,183783.53
4,cyber monday 2017,10.0,774,112439.99


---
**That's the pipeline.** Raw → Staging → Curated, with audits so we know the numbers are safe.

The cell below runs the same pipeline as a standalone script (`build.py`) and prints a checksum of the curated table, so anyone can confirm they rebuilt exactly the same result.


In [16]:
!python build.py

BUILD OK: curated_revenue_fact rows = 97887
run_id=2026-11-22T00:53:55
curated_checksum=8dd3e47715074f17a00ba9a709cdb96f
ALL CHECKS PASSED


---
# Going further (added after submission)

The sections below were added when this project was turned into a portfolio piece. They use the same curated table and answer two follow-up questions.

## A. How wrong would a naive join have been?

Here we join orders to the **raw** campaign file (one row per channel) instead of the deduplicated dimension, then compare campaign revenue with the curated answer.


In [17]:
naive_vs_curated = q("""
SELECT 'naive (raw campaigns, one row per channel)' AS method,
       COUNT(*) AS campaign_rows,
       ROUND(SUM(f.order_revenue), 2) AS campaign_revenue
FROM curated.curated_revenue_fact f
JOIN raw_campaigns r ON f.campaign_id = r.campaign_id
UNION ALL
SELECT 'curated (one row per order)',
       COUNT(*),
       ROUND(SUM(order_revenue), 2)
FROM curated.curated_revenue_fact
WHERE campaign_id IS NOT NULL
""")
display(naive_vs_curated)
naive, curated = naive_vs_curated["campaign_revenue"].tolist()
print(f"The naive join overstates campaign revenue by {naive / curated - 1:.1%}")


,method,campaign_rows,campaign_revenue
0,"naive (raw campaigns, one row per channel)",12861,1938435.54
1,curated (one row per order),7016,1045579.28


The naive join overstates campaign revenue by 85.4%


## B. A fairer comparison: revenue per day

Campaign windows cover only 31 days, so comparing **total** revenue in and out of campaigns isn't meaningful. Below we compare the **average day** inside and outside the windows, from January 2017 to August 2018 (the months where the store has steady volume).


In [18]:
per_day = q("""
WITH daily AS (
    SELECT CAST(order_purchase_timestamp AS DATE) AS day,
           campaign_id IS NOT NULL AS in_campaign,
           COUNT(*) AS orders,
           SUM(order_revenue) AS revenue
    FROM curated.curated_revenue_fact
    WHERE order_purchase_timestamp >= '2017-01-01'
      AND order_purchase_timestamp <  '2018-09-01'
    GROUP BY 1, 2
)
SELECT CASE WHEN in_campaign THEN 'campaign days' ELSE 'other days' END AS segment,
       COUNT(*) AS days,
       ROUND(AVG(orders), 1) AS orders_per_day,
       ROUND(AVG(revenue), 2) AS revenue_per_day,
       ROUND(SUM(revenue) / SUM(orders), 2) AS avg_order_value
FROM daily
GROUP BY 1
ORDER BY 1
""")
display(per_day)


,segment,days,orders_per_day,revenue_per_day,avg_order_value
0,campaign days,31,226.3,33728.36,149.03
1,other days,571,158.7,25520.88,160.86


### What this tells us

- On campaign days the store takes **more orders per day** and **more revenue per day** than on other days.
- But the **average order is smaller** on campaign days, which fits shoppers buying discounted items.

## C. Campaign by campaign: is every window a lift?

The average above hides big differences. Here each window is compared with the **four weeks just before it**.


In [19]:
by_window = q("""
WITH daily AS (
    SELECT CAST(order_purchase_timestamp AS DATE) AS dt, SUM(order_revenue) AS revenue
    FROM curated.curated_revenue_fact
    GROUP BY 1
)
SELECT c.campaign_name,
       ROUND((SELECT SUM(revenue) FROM daily WHERE dt BETWEEN c.window_start AND c.window_end)
             / (DATEDIFF('day', c.window_start, c.window_end) + 1), 0) AS revenue_per_day_in_window,
       ROUND((SELECT SUM(revenue) FROM daily WHERE dt BETWEEN c.window_start - 28 AND c.window_start - 1)
             / 28, 0) AS revenue_per_day_prior_4_weeks
FROM curated.campaigns_dim c
WHERE c.window_start < DATE '2018-09-01'
ORDER BY c.window_start
""")
by_window["change"] = (by_window["revenue_per_day_in_window"] / by_window["revenue_per_day_prior_4_weeks"] - 1).map("{:+.0%}".format)
display(by_window)


,campaign_name,revenue_per_day_in_window,revenue_per_day_prior_4_weeks,change
0,black friday 2017,100804.0,27878.0,+262%
1,cyber monday 2017,56220.0,36460.0,+54%
2,christmas 2017,21727.0,45176.0,-52%
3,new year 2018,23105.0,29791.0,-22%
4,carnival 2018,30631.0,35137.0,-13%


- **Black Friday 2017** is the only clear spike: revenue per day was more than **3.5 times** the previous four weeks.
- **Cyber Monday** is also above its baseline, partly because shoppers were still buying after Black Friday.
- **Christmas, New Year and Carnival** sit **below** the weeks before them. A likely reason is that people order gifts early so they arrive in time, and fewer orders are placed during the holidays themselves.

So the "campaign lift" in section B is almost entirely **Black Friday**. Reporting one average lift for all campaigns would be misleading.

### Caveats (read before quoting these numbers)

- The campaign file is **synthetic**: Olist did not publish its promotions, so the windows are placed on real holidays (Black Friday, Christmas and so on). The lift shown here is really **holiday seasonality**, not proof that a campaign caused it.
- **Black Friday 2018** matches no orders because the dataset ends in early September 2018.
- `order_revenue` is item price plus freight; it ignores the discount percentage, since the data doesn't record which orders used a coupon.
